<a href="https://colab.research.google.com/github/Shineii86/PullShark/blob/main/notebooks/PullShark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/PullShark/refs/heads/main/images/PullShark.png" width="280px" alt="Pull Shark Achievement">
  <h1>🦈 GitHub Pull Shark Automator</h1>
  <p><b>Automate pull request creation and merging to unlock the Pull Shark achievement.</b></p>
  <p>
    <img src="https://img.shields.io/badge/Python-3.8%2B-3776AB?logo=python&logoColor=white" alt="Python">
    <img src="https://img.shields.io/badge/License-MIT-yellow" alt="License">
    <img src="https://img.shields.io/badge/PRs-Automated-28a745" alt="Automated">
  </p>
</div>

---

<table>
  <tr>
    <td>
      <h3>📋 How It Works</h3>
      <ol>
        <li>Creates a <b>new branch</b> from your base branch</li>
        <li>Makes a <b>small commit</b> (updates README.md)</li>
        <li>Opens a <b>pull request</b> to the base branch</li>
        <li><b>Merges</b> the PR automatically</li>
        <li>Repeats for the number of PRs you specify</li>
      </ol>
    </td>
    <td>
      <h3>🦈 Pull Shark Tiers</h3>
      <table>
        <tr><td>🥉 <b>Default</b></td><td>2 PRs</td></tr>
        <tr><td>🥈 <b>Bronze</b></td><td>16 PRs</td></tr>
        <tr><td>🥇 <b>Silver</b></td><td>128 PRs</td></tr>
        <tr><td>🏆 <b>Gold</b></td><td>1024 PRs</td></tr>
      </table>
    </td>
  </tr>
</table>

<div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 12px 16px; border-radius: 4px; margin: 8px 0;">
  <b>⚠️ Important — Read Before Running</b>
  <ul style="margin: 8px 0 0 0;">
    <li>This script creates and merges <b>real pull requests</b> on your GitHub account.</li>
    <li><b>Never share your Personal Access Token</b> — treat it like a password.</li>
    <li>Use a <b>dedicated test repository</b> to avoid cluttering important projects.</li>
    <li>GitHub may rate-limit excessive API calls; the built-in delay helps prevent this.</li>
  </ul>
</div>

In [ ]:
#@title 📦 Step 1 — Install Dependencies & Load Package
#@markdown <small>Installs PyGithub and loads the PullShark package. Run this cell first.</small>

import sys, os, time
from IPython.display import display, HTML

display(HTML('''
<div style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 10px 14px; border-radius: 4px; margin: 4px 0; font-family: sans-serif;">
  ⏳ Installing dependencies and loading PullShark...
</div>
'''))

!pip install -q PyGithub

# If running from the repo root, add it to path so we can import pullshark
if os.path.exists('pullshark'):
    sys.path.insert(0, '.')
else:
    # Colab: clone the repo and add to path
    !git clone -q https://github.com/Shineii86/PullShark.git /tmp/pullshark-repo
    sys.path.insert(0, '/tmp/pullshark-repo')

from pullshark.config import Config
from pullshark.core import PullSharkBot
from pullshark import __version__

display(HTML(f'''
<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px 14px; border-radius: 4px; margin: 4px 0; font-family: sans-serif;">
  ✅ <b>PullShark v{__version__}</b> loaded successfully! You can now configure and run the bot below.
</div>
'''))

In [ ]:
#@title 🔌 Step 2 — Test Connection
#@markdown <small>Validates your token, repo access, and API quota. Run before your first automation.</small>

TEST_TOKEN = ""  #@param {type:"string"}
TEST_USERNAME = ""  #@param {type:"string"}
TEST_REPO = ""  #@param {type:"string"}

from github import Github
from IPython.display import display, HTML

def status_icon(ok):
    return '✅' if ok else '❌'

def check(label, ok, detail=''):
    color = '#28a745' if ok else '#dc3545'
    extra = f' <span style="color: #666;">— {detail}</span>' if detail else ''
    display(HTML(f'<div style="font-family: monospace; padding: 2px 0; color: {color};">{status_icon(ok)} {label}{extra}</div>'))

if not all([TEST_TOKEN, TEST_USERNAME, TEST_REPO]):
    display(HTML('<div style="color: #856404; background-color: #fff3cd; padding: 8px 12px; border-radius: 4px; font-family: sans-serif;">⚠️ Fill in all three fields above and run this cell to test your connection.</div>'))
else:
    display(HTML('<div style="font-family: sans-serif; padding: 4px 0;"><b>🔌 Testing connection...</b></div>'))
    
    # Test 1: Token validity
    try:
        g = Github(TEST_TOKEN)
        user = g.get_user()
        actual_username = user.login
        check('Token valid', True, f'Authenticated as @{actual_username}')
        token_ok = True
    except Exception as e:
        check('Token valid', False, str(e)[:80])
        token_ok = False

    # Test 2: Username match
    if token_ok:
        username_match = actual_username.lower() == TEST_USERNAME.lower()
        check('Username matches token', username_match, f'Expected @{TEST_USERNAME}, got @{actual_username}' if not username_match else '')

    # Test 3: Repo access
    repo_ok = False
    if token_ok:
        try:
            repo = g.get_user().get_repo(TEST_REPO)
            check('Repository accessible', True, f'{repo.full_name} ({"private" if repo.private else "public"})')
            repo_ok = True
        except Exception as e:
            check('Repository accessible', False, str(e)[:80])

    # Test 4: Write access
    if token_ok and repo_ok:
        try:
            base_sha = repo.get_branch(repo.default_branch).commit.sha
            test_ref = repo.create_git_ref('refs/heads/_pullshark_test_', base_sha)
            test_ref.delete()
            check('Write access confirmed', True)
        except Exception as e:
            check('Write access confirmed', False, str(e)[:80])

    # Test 5: API Rate Limit
    if token_ok:
        rate = g.get_rate_limit()
        remaining = rate.core.remaining
        limit = rate.core.limit
        reset = rate.core.reset.strftime('%H:%M:%S')
        enough = remaining > 20
        check('API rate limit', enough, f'{remaining}/{limit} remaining (resets {reset})')

    # Test 6: Auto-pr branches
    if token_ok and repo_ok:
        auto_branches = [b.name for b in repo.get_branches() if b.name.startswith('auto-pr-')]
        if auto_branches:
            check('Existing auto-pr branches', False, f'{len(auto_branches)} found — run Step 5 to clean up')
        else:
            check('Existing auto-pr branches', True, 'None found — repo is clean')

    all_ok = token_ok and repo_ok
    if all_ok:
        display(HTML('<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px 14px; border-radius: 4px; margin: 8px 0 4px 0; font-family: sans-serif;"><b>🎉 All checks passed!</b> You\'re ready to run the automation in Step 3.</div>'))
    else:
        display(HTML('<div style="background-color: #f8d7da; border-left: 4px solid #dc3545; padding: 10px 14px; border-radius: 4px; margin: 8px 0 4px 0; font-family: sans-serif;"><b>⚠️ Some checks failed.</b> Fix the issues above before proceeding to Step 3.</div>'))

In [ ]:
#@title 🔍 Step 3 — Dry Run (Preview)
#@markdown <small>See exactly what the bot will do without making any changes to GitHub. <b>Recommended before your first real run.</b></small>

#@markdown ### Credentials
DRY_USERNAME = "Shineii86"  #@param {type:"string"}
DRY_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

#@markdown ### Settings
DRY_REPO = "PullShark"  #@param {type:"string"}
DRY_NUM_PRS = 4  #@param {type:"slider", min:2, max:50, step:1}
DRY_MERGE_METHOD = "merge"  #@param ["merge", "squash", "rebase"]

from IPython.display import display, HTML

display(HTML('''
<div style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
  🔍 <b>Dry Run Mode</b> — previewing what would happen. No changes will be made.
</div>
'''))

config = Config(
    github_username=DRY_USERNAME,
    github_token=DRY_TOKEN,
    repo_name=DRY_REPO,
    num_prs=DRY_NUM_PRS,
    dry_run=True,
    merge_method=DRY_MERGE_METHOD,
)

errors = config.validate()
if errors:
    display(HTML('<div style="background-color: #f8d7da; border-left: 4px solid #dc3545; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;"><b>❌ Configuration errors:</b><ul style="margin: 6px 0 0 0;">' + ''.join(f'<li>{e}</li>' for e in errors) + '</ul></div>'))
else:
    bot = PullSharkBot(config)
    bot.run()
    display(HTML('''
    <div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
      ✅ <b>Dry run complete!</b> If everything looks right, proceed to Step 4 for the real run.
    </div>
    '''))

In [ ]:
#@title 🚀 Step 4 — Run for Real
#@markdown <small>Fill in your details and run to create and merge PRs. <b>This makes real changes!</b></small>

# =============================================================================
#  🔧 Configuration
# =============================================================================

#@markdown ### GitHub Credentials
GITHUB_USERNAME = "Shineii86"  #@param {type:"string"}
GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

#@markdown ### Repository
REPO_NAME = "PullShark"  #@param {type:"string"}
BASE_BRANCH = "main"  #@param {type:"string"}

#@markdown ### Automation Settings
NUM_PRS = 4  #@param {type:"slider", min:2, max:50, step:1}
DELAY_SECONDS = 10  #@param {type:"slider", min:5, max:60, step:5}
MAX_RETRIES = 3  #@param {type:"slider", min:1, max:10, step:1}
MERGE_METHOD = "merge"  #@param ["merge", "squash", "rebase"]

#@markdown ### Pre-flight
CHECK_RATE_LIMIT = True  #@param {type:"boolean"}

# =============================================================================
#  🎬 Pre-flight Summary
# =============================================================================

from IPython.display import display, HTML
import time as _time

config = Config(
    github_username=GITHUB_USERNAME,
    github_token=GITHUB_TOKEN,
    repo_name=REPO_NAME,
    num_prs=NUM_PRS,
    base_branch=BASE_BRANCH,
    delay_seconds=DELAY_SECONDS,
    max_retries=MAX_RETRIES,
    merge_method=MERGE_METHOD,
)

errors = config.validate()
if errors:
    display(HTML('<div style="background-color: #f8d7da; border-left: 4px solid #dc3545; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;"><b>❌ Configuration errors:</b><ul style="margin: 6px 0 0 0;">' + ''.join(f'<li>{e}</li>' for e in errors) + '</ul></div>'))
else:
    bot = PullSharkBot(config)

    # Rate limit check
    if CHECK_RATE_LIMIT:
        try:
            info = bot.check_rate_limit()
            rate_color = '#28a745' if info['enough_for_run'] else '#ffc107'
            rate_icon = '✅' if info['enough_for_run'] else '⚠️'
            display(HTML(f'''
            <div style="background-color: #f8f9fa; border-left: 4px solid {rate_color}; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
              {rate_icon} <b>API Quota:</b> {info["remaining"]}/{info["limit"]} calls remaining (resets {info["reset"]})
            </div>
            '''))
            if not info['enough_for_run']:
                display(HTML('<div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">⚠️ <b>Low API quota.</b> Consider reducing PR count or waiting for the rate limit to reset.</div>'))
        except Exception as e:
            display(HTML(f'<div style="color: #856404; background-color: #fff3cd; padding: 8px 12px; border-radius: 4px; font-family: sans-serif;">⚠️ Could not check rate limit: {str(e)[:60]}</div>'))

    # Config summary
    display(HTML(f'''
    <div style="background-color: #f8f9fa; border: 1px solid #dee2e6; border-radius: 6px; padding: 14px 18px; margin: 8px 0; font-family: sans-serif;">
      <b>📋 Configuration Summary</b>
      <table style="margin-top: 8px; font-size: 14px;">
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">User</td><td><code>@{GITHUB_USERNAME}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Repository</td><td><code>{REPO_NAME}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Base Branch</td><td><code>{BASE_BRANCH}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Pull Requests</td><td><b>{NUM_PRS}</b></td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Merge Method</td><td><code>{MERGE_METHOD}</code></td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Delay</td><td>{DELAY_SECONDS}s between PRs</td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Max Retries</td><td>{MAX_RETRIES} per merge</td></tr>
        <tr><td style="padding: 2px 12px 2px 0; color: #666;">Est. Time</td><td>~{NUM_PRS * (DELAY_SECONDS + 8)}s ({(NUM_PRS * (DELAY_SECONDS + 8)) // 60}m {(NUM_PRS * (DELAY_SECONDS + 8)) % 60}s)</td></tr>
      </table>
    </div>
    '''))

    # Confirm
    display(HTML('''
    <div style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
      🚀 <b>Starting automation...</b> Watch the output below for progress.
    </div>
    '''))

    start_time = _time.time()
    successful = bot.run()
    elapsed = int(_time.time() - start_time)
    mins, secs = divmod(elapsed, 60)

    # --- Results Summary ---
    success = successful >= 2
    bg = '#d4edda' if success else '#f8d7da'
    border = '#28a745' if success else '#dc3545'
    icon = '🎉' if success else '⚠️'
    title = 'Achievement Unlocked!' if success else 'More PRs Needed'
    tier = 'Gold 🏆' if successful >= 1024 else 'Silver 🥇' if successful >= 128 else 'Bronze 🥈' if successful >= 16 else 'Default 🥉' if successful >= 2 else 'None'

    display(HTML(f'''
    <div style="background-color: {bg}; border: 2px solid {border}; border-radius: 8px; padding: 16px 20px; margin: 12px 0; font-family: sans-serif;">
      <div style="font-size: 20px; margin-bottom: 8px;">{icon} <b>{title}</b></div>
      <table style="font-size: 14px;">
        <tr><td style="padding: 2px 16px 2px 0; color: #666;">Merged PRs</td><td><b>{successful}</b> / {NUM_PRS}</td></tr>
        <tr><td style="padding: 2px 16px 2px 0; color: #666;">Pull Shark Tier</td><td><b>{tier}</b></td></tr>
        <tr><td style="padding: 2px 16px 2px 0; color: #666;">Merge Method</td><td>{MERGE_METHOD}</td></tr>
        <tr><td style="padding: 2px 16px 2px 0; color: #666;">Time Elapsed</td><td>{mins}m {secs}s</td></tr>
      </table>
    </div>
    '''))

    if success:
        display(HTML('''
        <div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
          💡 <b>Next step:</b> Go to your <a href="https://github.com" target="_blank">GitHub profile</a> and look for the Pull Shark badge. It may take a few minutes to appear.
          <br>🧹 <b>Cleanup:</b> Run Step 5 below to delete the auto-created branches.
        </div>
        '''))

In [ ]:
#@title 🧹 Step 5 — Cleanup Branches
#@markdown <small>Delete all <code>auto-pr-*</code> branches created by the bot. Use dry-run first to see what would be deleted.</small>

CLEAN_TOKEN = ""  #@param {type:"string"}
CLEAN_USERNAME = ""  #@param {type:"string"}
CLEAN_REPO = ""  #@param {type:"string"}
CLEAN_DRY_RUN = True  #@param {type:"boolean"}

from IPython.display import display, HTML

if not all([CLEAN_TOKEN, CLEAN_USERNAME, CLEAN_REPO]):
    display(HTML('<div style="color: #856404; background-color: #fff3cd; padding: 8px 12px; border-radius: 4px; font-family: sans-serif;">⚠️ Fill in all three fields above and run this cell.</div>'))
else:
    mode_label = 'PREVIEW (dry run)' if CLEAN_DRY_RUN else 'LIVE DELETION'
    mode_color = '#2196F3' if CLEAN_DRY_RUN else '#dc3545'
    display(HTML(f'''
    <div style="background-color: #f8f9fa; border-left: 4px solid {mode_color}; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">
      🧹 <b>Branch Cleanup</b> — mode: <b>{mode_label}</b>
    </div>
    '''))

    config = Config(
        github_username=CLEAN_USERNAME,
        github_token=CLEAN_TOKEN,
        repo_name=CLEAN_REPO,
    )
    bot = PullSharkBot(config)
    deleted = bot.clean(dry_run=CLEAN_DRY_RUN)

    if deleted == 0:
        display(HTML('<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">✨ <b>Repo is clean!</b> No auto-pr branches found.</div>'))
    elif CLEAN_DRY_RUN:
        display(HTML(f'<div style="background-color: #e8f4fd; border-left: 4px solid #2196F3; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">📋 <b>{deleted} branch(es) would be deleted.</b> Uncheck dry-run above and re-run to delete them.</div>'))
    else:
        display(HTML(f'<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 10px 14px; border-radius: 4px; margin: 8px 0; font-family: sans-serif;">🗑️ <b>{deleted} branch(es) deleted!</b> Repo is clean.</div>'))

---

<details>
<summary><b>📚 Troubleshooting & Help</b></summary>

| Issue | Solution |
|:------|:---------|
| `BadCredentialsException` | Token is wrong or expired. Generate a new one with `repo` scope. |
| `405 Not mergeable` | Enable **Allow auto-merge** in repo Settings → General → Pull Requests. |
| Hangs at "Waiting for mergeability" | PR may have a conflict. Delete the branch manually and retry. |
| `RateLimitExceededException` | Wait an hour or increase `DELAY_SECONDS`. |
| No badge after success | Wait a few minutes and refresh your GitHub profile. |

</details>

<details>
<summary><b>🔐 How to Get a Personal Access Token</b></summary>

1. Go to **Settings** → **Developer settings** → **Personal access tokens** → **Tokens (classic)**.
2. Click **Generate new token (classic)**.
3. Name it (e.g., `Pull Shark Bot`).
4. Check the **`repo`** scope.
5. Click **Generate token** and copy it immediately.

</details>

<details>
<summary><b>🔀 Merge Methods Explained</b></summary>

| Method | What it does |
|:-------|:-------------|
| `merge` | Creates a merge commit. Preserves all branch history. Default behavior. |
| `squash` | Combines all commits into one. Cleaner history, single commit per PR. |
| `rebase` | Replays commits on base branch. Linear history, no merge commit. |

</details>

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for GitHub achievement hunters*

***Found this useful? Give it a Star! ⭐ [Shineii86/PullShark](https://github.com/Shineii86/PullShark)***

</div>